# AMICA on the EEGBCI dataset — a hands-on demo

**For:** Annalisa

This notebook walks through fitting **AMICA** (Adaptive Mixture ICA, Palmer et al. 2011) on a small slice of the [EEG Motor Movement/Imagery Dataset](https://mne.tools/stable/generated/mne.datasets.eegbci.load_data.html) (PhysioNet EEGBCI), then inspecting the resulting components with topomaps, time series, and PSDs.

## What you'll get out of this

1. A clean Raw object from one EEGBCI subject (64 channels, 10–10 montage)
2. A fitted MNE `ICA` object produced by `py_amica.fit_ica`
3. Component **topomaps**, **time courses**, and **PSDs**
4. A cleaned Raw object after removing artifact components
5. A before/after PSD comparison

## Requirements

```
pip install pyamica[mne,jax]   # or: pip install -e .[all] from the repo root
```

GPU is optional. On CPU the fit takes a few minutes for 64 channels; on a single GPU it's seconds.

## What AMICA is, in one paragraph

AMICA fits an ICA *mixture* model where every source has its own mixture of generalized Gaussians with learned shape (`rho`), scale (`beta`), and location (`mu`). The shape parameter slides between Laplacian (super-Gaussian, `rho=1`) and Gaussian (`rho=2`), so each component adapts to its actual distribution rather than assuming one. Optimization uses natural gradient followed by Newton steps (Palmer et al. 2008). In Delorme et al. (2012) AMICA ranked first of 22 ICA algorithms for EEG on mutual-information reduction and source dipolarity.

## 1. Setup

In [ ]:
import os

# Optional: ask AMICA to use JAX (GPU if available, CPU fallback otherwise).
# Set to "1" if you want to force the pure-NumPy backend.
os.environ.setdefault("AMICA_NO_JAX", "0")

import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

import matplotlib.pyplot as plt
import mne
from mne.datasets import eegbci

from py_amica import fit_ica

mne.set_log_level("WARNING")
print("MNE:", mne.__version__)

# Quick check on which backend AMICA will use
from py_amica.backend import HAS_JAX

print("AMICA backend:", "JAX (GPU/CPU)" if HAS_JAX else "NumPy (CPU)")

## 2. Load the EEGBCI data

We'll use **subject 1**, runs **4, 8, 12** — the *motor-imagery hands* runs (~2 min each ≈ 6 min total). The first time you run this, MNE will download ~10 MB from PhysioNet into your `~/mne_data` folder; subsequent runs are cached.

Two things to know about EEGBCI:

- Channel names come from PhysioNet with trailing dots (`"Fc5."`, `"Fcz."`). `eegbci.standardize` strips them.
- The data is in **EDF** and lives in microvolt-scale; MNE loads it in volts.

In [ ]:
subject = 1
runs = [4, 8, 12]  # motor imagery: left/right hand

raw_fnames = eegbci.load_data(subject, runs, update_path=True)
raws = [mne.io.read_raw_edf(f, preload=True) for f in raw_fnames]
raw = mne.concatenate_raws(raws)

# Clean channel names: "Fc5." -> "FC5", apply standard 10-10 montage
eegbci.standardize(raw)
montage = mne.channels.make_standard_montage("standard_1005")
raw.set_montage(montage)

print(raw)
print("Channels:", len(raw.ch_names))
print("Duration: {:.1f} s @ {:.0f} Hz".format(raw.n_times / raw.info["sfreq"], raw.info["sfreq"]))

### Preprocess: high-pass + average reference

Two reasons to high-pass before ICA:
- ICA assumes stationarity; slow drifts violate that.
- AMICA (and any ICA) decomposes drifts into one or two components that swamp the cortical sources.

1 Hz is a good default for ICA-targeted preprocessing (see Winkler et al. 2015).

In [ ]:
raw.filter(l_freq=1.0, h_freq=40.0, fir_design="firwin")
raw.set_eeg_reference("average", projection=False)
raw.notch_filter(freqs=[60.0])  # US 60 Hz line noise (PhysioNet recordings)
raw

### Quick look at the raw data + PSD

In [ ]:
fig = raw.compute_psd(fmax=60).plot(show=False)
fig.suptitle("Raw PSD (after 1-40 Hz band-pass + 60 Hz notch)")
plt.show()

In [ ]:
# Visual inspection of the first 10 seconds
raw.plot(duration=10, n_channels=20, scalings={"eeg": 50e-6}, show=True);

## 3. Fit AMICA

`py_amica.fit_ica` wraps the full pipeline (pre-whitening + PCA + AMICA unmixing) and returns a **standard `mne.preprocessing.ICA` object** — so every MNE ICA visualizer, `apply`, `find_bads_eog/ecg`, etc. work unchanged.

### Key knobs

| Parameter | What it does | Suggested |
|---|---|---|
| `n_components` | PCA dim before ICA. EEGBCI has 64 channels; 30–40 is a good default for motor data | 32 |
| `max_iter` | EM iterations. 2000 is the AMICA paper default | 2000 |
| `num_mix` | Generalized-Gaussian mixture components per source | 3 |
| `random_state` | Seed for reproducibility | 42 |

On CPU this takes a few minutes; on a single H100/A100/T4 GPU it's seconds.

**Quick mode:** if you just want to see the pipeline run end-to-end, drop `max_iter` to 200. You'll get something usable but undertrained — don't read anything into the components.

In [ ]:
import time

N_COMPONENTS = 32
MAX_ITER = 2000  # drop to 200 for a fast sanity run
RANDOM_STATE = 42

t0 = time.perf_counter()
ica = fit_ica(
    raw,
    n_components=N_COMPONENTS,
    max_iter=MAX_ITER,
    num_mix=3,
    random_state=RANDOM_STATE,
)
elapsed = time.perf_counter() - t0

print(f"AMICA fit completed in {elapsed:.1f} s ({ica.n_iter_} iterations)")
ica

## 4. Inspect the components

`ica` is a regular MNE ICA object, so we can use all of MNE's component visualizers.

### 4.1 Topomaps for all components

In [ ]:
ica.plot_components(show=True);

### 4.2 Component time courses

Click on any component time series in the interactive view to mark it for exclusion (its panel label turns red). For motor-imagery data, look for:
- **Frontal blink-like topographies** + sharp deflections in the time course → eye blinks
- **Lateral frontal dipolar topographies** + slow drifts → horizontal eye movements
- **Temporal/lateral topographies with high-frequency bursts** → muscle (EMG)
- **Localized topography with isolated spikes** → single-channel noise

In [ ]:
ica.plot_sources(raw, show=True);

### 4.3 Per-component properties (topo + spectrum + epoch image)

Look at the first 8 components in detail. The PSD lets you spot line noise (50/60 Hz peak) and muscle (broadband high-frequency rise).

In [ ]:
ica.plot_properties(raw, picks=list(range(8)), show=True);

## 5. Detect artifact components automatically

EEGBCI has **no dedicated EOG or ECG channels**, so we use frontal EEG channels as a proxy for blink-correlated activity. This is approximate — for proper artifact rejection on your own data, prefer real EOG/ECG channels.

In [ ]:
# Use Fp1 as a blink proxy. Channel name after eegbci.standardize is "Fp1".
eog_proxy = "Fp1"

eog_indices, eog_scores = ica.find_bads_eog(raw, ch_name=eog_proxy, threshold=2.5)
print(f"Blink-like components (Fp1 correlation, threshold 2.5 SD): {eog_indices}")

fig = ica.plot_scores(eog_scores, exclude=eog_indices, show=False)
fig.suptitle("Per-component correlation with Fp1 (EOG proxy)")
plt.show()

In [ ]:
# Detailed look at flagged components
if len(eog_indices) > 0:
    ica.plot_properties(raw, picks=eog_indices, show=True)
else:
    print("No components flagged — threshold may be too strict, try lowering to 2.0.")

## 6. Clean the data

We mark the blink-like components in `ica.exclude` and call `ica.apply(...)` to back-project the remaining components onto the channel space.

In [ ]:
ica.exclude = list(eog_indices)
print("Excluded components:", ica.exclude)

raw_clean = ica.apply(raw.copy())

## 7. Before vs after

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

raw.compute_psd(fmax=40).plot(axes=axes[0], show=False)
axes[0].set_title("Raw — before ICA cleaning")

raw_clean.compute_psd(fmax=40).plot(axes=axes[1], show=False)
axes[1].set_title(f"Raw — after ICA cleaning ({len(ica.exclude)} components removed)")

plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side time-domain comparison of the same 10 s window
raw.plot(duration=10, n_channels=20, scalings={"eeg": 50e-6}, title="Before ICA", show=True)
raw_clean.plot(duration=10, n_channels=20, scalings={"eeg": 50e-6}, title="After ICA", show=True);

## 8. Save the results

The fitted ICA object is a standard MNE one, so MNE's I/O works unchanged. Saving the cleaned Raw is optional — typically you keep the raw + ICA and re-apply on the fly.

In [ ]:
from pathlib import Path

out_dir = Path("./amica_eegbci_results")
out_dir.mkdir(exist_ok=True)

ica.save(out_dir / f"sub-{subject:02d}_amica-ica.fif", overwrite=True)
raw_clean.save(out_dir / f"sub-{subject:02d}_cleaned_raw.fif", overwrite=True)

print("Saved to:", out_dir.resolve())
for f in out_dir.iterdir():
    print("  ", f.name)

## 9. Where to go next

- **Try another subject:** change `subject = 1` to anything 1–109.
- **Try different runs:** runs 1–2 = baseline (eyes open/closed), 3/7/11 = real hand motion, 4/8/12 = imagined hand motion, 5/9/13 = real hands/feet, 6/10/14 = imagined hands/feet.
- **Compare to Picard/Infomax:** swap `fit_ica(...)` for `mne.preprocessing.ICA(method="picard").fit(raw)` to see how AMICA's components differ.
- **Outlier rejection:** AMICA supports per-sample outlier downweighting (Klug et al. 2024) via `fit_params={"do_reject": True, "rejsig": 3.0}` passed to `fit_ica`.
- **Epoch-locked analysis:** for the motor-imagery paradigm, build epochs around T1/T2 annotations and look at component activations time-locked to cue onset.

## References

- Palmer, Kreutz-Delgado, Makeig (2011). *AMICA: An Adaptive Mixture of Independent Component Analyzers with Shared Components.* SCCN Technical Report.
- Delorme, Palmer, Onton, Oostenveld, Makeig (2012). *Independent EEG sources are dipolar.* PLoS ONE.
- Schalk, McFarland, Hinterberger, Birbaumer, Wolpaw (2004). *BCI2000: A General-Purpose Brain-Computer Interface (BCI) System.* IEEE TBME. (EEGBCI dataset citation)
- Klug, Berg, Gramann (2024). *Optimizing EEG ICA decomposition.* Sci. Reports.